# Week 07A — Integration Lab: An End-to-End Inspection Station (and Midterm Revision)

**MCTA 4364 Machine Vision** · Session 1 of Week 7 · 80 minutes · Session B is the **midterm** (Weeks 1–6)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W07_integration/W07A_integration_lab_revision.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W07_integration/W07A_integration_lab_revision.ipynb)

In six weeks you have met every building block of a classical and a learned vision system. Today you **put them together** on one realistic
task: automatic inspection of a machined **flange** on a conveyor. You will build the system twice — once with hand-engineered rules and once with
a CNN — compare them like an engineer (accuracy, speed, data needs, robustness, explainability), and finish with a structured revision for the midterm.

### Learning outcomes
By the end of this session you will be able to:
1. Turn a **specification** (dimensions, tolerances, defect list, throughput) into a vision **pipeline**.
2. Combine acquisition, pre-processing, segmentation, contour analysis, geometric alignment and measurement **in millimetres** (Weeks 1–5).
3. Train a CNN classifier (Week 6) and show why a **hybrid** pipeline (classical localisation + learned decision) beats an end-to-end CNN on small data.
4. Evaluate systems with confusion matrices, timing and a **robustness test** under changed lighting.
5. Connect every topic of Weeks 1–6 in a concept map and self-test for the midterm.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1 | The brief: specification, pipeline design |
| 8–25 | 2–4 | Acquisition, segmentation, pose estimation, golden template |
| 25–40 | 5 | Measurement and rules; exercises E1–E3 in pairs |
| 40–55 | 6–7 | CNN vs hybrid; comparison and robustness test |
| 55–80 | 9–10 | Midterm revision: concept map, formula sheet, revision quiz, practice questions |
| Home | 8 | Open exercise E4 and challenge |

**Hardware:** about 2 minutes on a laptop CPU, seconds on a GPU. The data are generated on the fly: no downloads.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import partsim
from cvhelpers import SMOKE, check, interact, pipeline_diagram, quiz, show
from dlhelpers import confusion_matrix, count_params, fit, hw_profile, plot_confusion, plot_history, predict, set_seed

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = HW["device"]
set_seed(0)
CLASSES = partsim.CLASSES
MM = partsim.MM_PER_PX
N_TRAIN, N_TEST, EPOCHS = (12, 8, 1) if SMOKE else (120, 40, 12)
print(f"classes: {CLASSES} | {N_TRAIN} training and {N_TEST} test parts per class | {EPOCHS} epochs")

## 1. The brief

A supplier stamps and machines steel flanges. Your system must inspect **every part** on the conveyor and reject defective ones.

| item | specification |
|---|---|
| outer diameter | **40.0 ± 0.4 mm**, with one orientation notch (3.0 mm wide, 2.0 mm deep) |
| holes | centre hole Ø12 mm; four bolt holes Ø4 mm on a Ø28 mm pitch circle, at 45° to the notch |
| defects to reject | **missing hole**, **scratch** (> 5 mm long), **chip** on the rim, **undersize** (outside tolerance) |
| camera | monochrome-equivalent, 256 × 256 px, telecentric-like, **0.2 mm/pixel** after calibration (Week 2) |
| pose | random rotation, ±1.6 mm position |
| lighting | ring light with some non-uniformity; exposure drifts over a shift |
| throughput | 10 parts/s → **< 100 ms per part** on the line PC |

Two kinds of error have **different costs**: a *false reject* wastes a good part; an *escape* (defective part accepted) can cause a field failure.
Always report both, not just accuracy.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["acquire\n(W01-W02)", "pre-process\n(W03)", "segment +\ncontours (W04)", "pose from\nfeatures (W04-W05)",
                  "align to golden\ntemplate (W03, W05)", "measure in mm\n+ rules (W02, W04)", "or: CNN on\naligned crop (W06)", "accept /\nreject"],
                 colors=["#f1f3f4", "#e8f0fe", "#e8f0fe", "#fef7e0", "#fef7e0", "#e6f4ea", "#fce8e6", "#f1f3f4"], ax=ax, fontsize=10)
ax.set_title("Figure 1.1 - The inspection pipeline and the week each stage comes from", fontweight="bold")
plt.show()

## 2. Acquisition: what the camera sees

`partsim.make_part(seed, defect)` is our "camera": it renders a physically plausible image (brushed metal, conveyor belt, non-uniform ring light,
sensor noise) from the drawing above. Look at the undersize parts: can **you** see the 1.4 mm difference? A good system must measure, not guess.

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(16, 16.5))
for r, c in enumerate(CLASSES):
    for k in range(5):
        im, info = partsim.make_part(100 * r + k, c)
        axes[r, k].imshow(im[..., ::-1]); axes[r, k].set_xticks([]); axes[r, k].set_yticks([])
        if info["defect_box"] is not None:
            x0, y0, x1, y1 = info["defect_box"]
            axes[r, k].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="#f29900", lw=1.5))
        if k == 0:
            axes[r, k].set_ylabel(c, fontsize=13)
        axes[r, k].set_title(f"D = {info['outer_d_mm']:.2f} mm", fontsize=9)
plt.suptitle("Figure 2.1 - Five parts per class (orange box: where the defect is)", fontweight="bold", y=0.995)
plt.tight_layout(); plt.show()

In [ ]:
img0, info0 = partsim.make_part(7, "ok")
g0 = cv2.cvtColor(img0, cv2.COLOR_BGR2GRAY)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
axes[0].imshow(g0, cmap="gray"); axes[0].axhline(20, color="#d93025"); axes[0].set_title("grey image; red line = profile row"); axes[0].axis("off")
axes[1].plot(g0[20], color="#d93025", label="belt only (row 20)"); axes[1].plot(g0[128], color="#1a73e8", label="through the part (row 128)")
axes[1].set_xlabel("x (px)"); axes[1].set_ylabel("grey level"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title("the belt brightness drifts: the ring light is not uniform")
axes[2].hist(g0.ravel(), 128, color="#5f6368"); axes[2].set_yscale("log"); axes[2].set_xlabel("grey level")
axes[2].set_title("bimodal histogram: belt vs metal -> Otsu will work (W01B, W04A)")
plt.suptitle("Figure 2.2 - Checking the image before designing the algorithm", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Pre-processing and segmentation (Weeks 3–4)

1. Gaussian blur to suppress sensor noise (W03B).
2. **Otsu** threshold: the belt/metal contrast is large compared with the illumination drift (W04A).
3. **Opening** to remove specks (W03B).
4. `findContours` with **`RETR_CCOMP`**: the outer boundary and its **holes** in one call (W04B).

In [ ]:
def segment(img):
    """Return (outer_contour, hole_contours, part_mask) for one part image."""
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    cs, h = cv2.findContours(b, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)
    h = h.reshape(-1, 4)                                     # OpenCV 4 returns (1, N, 4), OpenCV 5 (N, 4)
    outer = max([i for i in range(len(cs)) if h[i][3] == -1], key=lambda i: cv2.contourArea(cs[i]))
    holes = [cs[j] for j in range(len(cs)) if h[j][3] == outer and cv2.contourArea(cs[j]) > 20]
    mask = np.zeros(b.shape, np.uint8)
    cv2.drawContours(mask, [cs[outer]], -1, 255, -1)
    for hc in holes:
        cv2.drawContours(mask, [hc], -1, 0, -1)
    return cs[outer], holes, mask


img_c, _ = partsim.make_part(11, "chip")
outer, holes, mask = segment(img_c)
vis = img_c.copy()
cv2.drawContours(vis, [outer], -1, (0, 200, 0), 2)
cv2.drawContours(vis, holes, -1, (255, 120, 0), 2)
g = cv2.GaussianBlur(cv2.cvtColor(img_c, cv2.COLOR_BGR2GRAY), (5, 5), 0)
t, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
show(img_c, b, mask, vis, titles=["input (chipped part)", f"Otsu threshold = {t:.0f}", "part mask (holes removed)",
                                  f"outer contour (green) + {len(holes)} holes (blue)"], figsize=(20, 5))

## 4. Pose from features, and a golden template (Weeks 4–5)

To compare a part with a reference we must undo its rotation and translation. The part's own **features** give the pose:

- **centre**: the minimum enclosing circle of the outer contour;
- **orientation modulo 90°**: the four bolt holes sit at 45° + $k$·90° from the notch, so the circular mean of $4\theta_i$ gives the angle up to 90°
  (the same trick that makes a *missing* hole harmless);
- **which of the four**: probe the rim with a small **template of the notch** (points that must be cut away and points that must be metal).
  A random chip rarely matches the rectangular notch.

With the pose we warp every part to a canonical position (W03A), and average 20 good parts into a **golden template**. The **difference** from the
golden template highlights any deviation: this *golden-template comparison* is a classic industrial technique. The broad red/blue shading in the difference images is the remaining **lighting
gradient**; the defects are the small, sharp structures.

In [ ]:
NOTCH_PROBES = [(0, 0.6, 0), (0, 1.2, 0), (0, 1.7, 0), (-1.0, 0.8, 0), (1.0, 0.8, 0), (-1.0, 1.6, 0), (1.0, 1.6, 0),
                (-2.3, 0.8, 1), (2.3, 0.8, 1), (-2.3, 1.6, 1), (2.3, 1.6, 1), (0, 2.7, 1)]   # (lateral mm, depth mm, 0 = cut / 1 = metal)


def estimate_pose(img):
    """Return (centre (x, y), notch angle in degrees clockwise from 'up', segmentation)."""
    outer, holes, mask = segment(img)
    (cx, cy), r = cv2.minEnclosingCircle(outer)
    bolt_area_max = np.pi * (partsim.SPEC["bolt_d"] / MM) ** 2           # 4x the nominal bolt-hole area
    angles = []
    for hc in holes:
        if cv2.contourArea(hc) < bolt_area_max:
            m = cv2.moments(hc)
            angles.append(np.arctan2(m["m10"] / m["m00"] - cx, -(m["m01"] / m["m00"] - cy)))
    base = np.angle(np.exp(4j * (np.array(angles) - np.pi / 4)).mean()) / 4
    best, best_score = base, -1
    for k in range(4):
        a = base + k * np.pi / 2
        u, v = np.array([np.sin(a), -np.cos(a)]), np.array([np.cos(a), np.sin(a)])
        score = 0
        for lat, dep, expect in NOTCH_PROBES:
            x, y = np.array([cx, cy]) + u * (r - dep / MM) + v * lat / MM
            yi, xi = int(round(y)), int(round(x))
            inside = 0 <= yi < mask.shape[0] and 0 <= xi < mask.shape[1]
            score += int(inside and (mask[yi, xi] > 0) == expect)
        if score > best_score:
            best, best_score = a, score
    return (cx, cy), float(np.rad2deg(best) % 360), (outer, holes, mask)


def align(img, out=128, crop=208):
    """Rotate the part so the notch points up, centre it, crop crop x crop px and resize to out x out."""
    (cx, cy), a, _ = estimate_pose(img)
    M = cv2.getRotationMatrix2D((cx, cy), a, 1.0)
    M[:, 2] += [crop / 2 - cx, crop / 2 - cy]
    aligned = cv2.warpAffine(img, M, (crop, crop), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return cv2.resize(aligned, (out, out), interpolation=cv2.INTER_AREA)


errs = []
for s in range(10 if SMOKE else 100):
    c = CLASSES[s % len(CLASSES)]
    im, inf = partsim.make_part(1000 + s, c)
    errs.append((estimate_pose(im)[1] - inf["angle_deg"] + 180) % 360 - 180)
print(f"pose error on {len(errs)} parts: mean |error| {np.mean(np.abs(errs)):.2f} deg, max {np.max(np.abs(errs)):.2f} deg")

im, inf = partsim.make_part(21, "missing_hole")
(cx, cy), a, (outer, holes, mask) = estimate_pose(im)
vis = im.copy()
cv2.circle(vis, (int(cx), int(cy)), 3, (0, 0, 255), -1)
tip = (int(cx + 95 * np.sin(np.deg2rad(a))), int(cy - 95 * np.cos(np.deg2rad(a))))
cv2.arrowedLine(vis, (int(cx), int(cy)), tip, (0, 0, 255), 2, tipLength=0.08)
for lat, dep, e in NOTCH_PROBES:
    th = np.deg2rad(a); u, v = np.array([np.sin(th), -np.cos(th)]), np.array([np.cos(th), np.sin(th)])
    _, r = cv2.minEnclosingCircle(outer)
    p = np.array([cx, cy]) + u * (r - dep / MM) + v * lat / MM
    cv2.circle(vis, tuple(np.round(p).astype(int)), 2, (0, 200, 0) if e else (255, 0, 255), -1)
zoom = cv2.resize(vis[max(0, tip[1] - 30):tip[1] + 30, max(0, tip[0] - 30):tip[0] + 30], (240, 240), interpolation=cv2.INTER_NEAREST)
show(vis, zoom, align(im, 208), titles=[f"estimated notch angle {a:.1f} deg (true {inf['angle_deg']:.1f}), even with a hole missing",
                                         "notch probes: magenta = must be cut, green = must be metal", "aligned: notch up, centred"],
     figsize=(20, 6))

In [ ]:
golden_parts = [partsim.make_part(5000 + s, "ok")[0] for s in range(20)]
golden = np.median(np.stack([cv2.cvtColor(align(p), cv2.COLOR_BGR2GRAY) for p in golden_parts]).astype(np.float32), 0)


def aligned_gray(img):
    """Aligned 128 x 128 grey crop with its brightness normalised to the golden template (robust to exposure drift)."""
    a_ = cv2.cvtColor(align(img), cv2.COLOR_BGR2GRAY).astype(np.float32)
    return a_ * np.median(golden) / max(np.median(a_), 1.0)


fig, axes = plt.subplots(2, len(CLASSES) + 1, figsize=(22, 8.8))
axes[0, 0].imshow(golden, cmap="gray"); axes[0, 0].set_title("golden template\n(median of 20 good parts)")
axes[1, 0].axis("off")
for k, c in enumerate(CLASSES):
    a_ = aligned_gray(partsim.make_part(40 + k, c)[0])
    axes[0, k + 1].imshow(a_, cmap="gray"); axes[0, k + 1].set_title(f"{c}: aligned")
    axes[1, k + 1].imshow(a_ - golden, cmap="seismic", vmin=-80, vmax=80); axes[1, k + 1].set_title("difference from golden")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Figure 4.1 - Golden-template comparison: every defect type appears in the difference image (undersize = a thin ring)", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Measurement and rules (Weeks 2 and 4)

Each defect maps to a **measurable feature**:

| defect | feature | how |
|---|---|---|
| missing hole | number of holes (expect 5) | contour hierarchy |
| chip | number of rim **convexity defects** deeper than 0.8 mm (expect 1: the notch) | `convexHull` + `convexityDefects` |
| scratch | dark thin pixels on the metal | **black-hat** morphology inside the eroded part mask |
| undersize | outer diameter in **mm** | minimum enclosing circle × 0.2 mm/px |

The rules are then a short decision list, checked in order of severity.

In [ ]:
def features(img):
    outer, holes, mask = segment(img)
    (_, _), r = cv2.minEnclosingCircle(outer)
    d = cv2.convexityDefects(outer, cv2.convexHull(outer, returnPoints=False))
    d = np.zeros((0, 4)) if d is None else d.reshape(-1, 4)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    inner = cv2.erode(mask, np.ones((7, 7), np.uint8))
    blackhat = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
    return dict(diameter_mm=2 * r * MM, n_holes=len(holes), n_rim_defects=int((d[:, 3] / 256.0 > 0.8 / MM).sum()),
                scratch_px=int(((blackhat > 25) & (inner > 0)).sum()))


def decide_rules(f):
    if f["n_holes"] < 5:
        return "missing_hole"
    if f["n_rim_defects"] >= 2:
        return "chip"
    if f["scratch_px"] > 15:
        return "scratch"
    if abs(f["diameter_mm"] - 40.0) > 0.4:
        return "undersize"
    return "ok"


X_tr, y_tr, _ = partsim.make_dataset(N_TRAIN, seed=1)
X_te, y_te, _ = partsim.make_dataset(N_TEST, seed=2)
t0 = time.perf_counter()
F_te = [features(im) for im in X_te]
pred_rules = np.array([CLASSES.index(decide_rules(f)) for f in F_te])
ms_rules = (time.perf_counter() - t0) / len(X_te) * 1000
acc_rules = (pred_rules == y_te).mean()
print(f"rules: accuracy {acc_rules:.1%} on {len(y_te)} test parts, {ms_rules:.1f} ms per part, 0 training images needed")

In [ ]:
F_tr = [features(im) for im in X_tr]
fig, axes = plt.subplots(1, 4, figsize=(22, 4.2))
cols = plt.cm.tab10.colors
for ci, c in enumerate(CLASSES):
    sel = [f for f, y in zip(F_tr, y_tr) if y == ci]
    axes[0].hist([f["diameter_mm"] for f in sel], np.linspace(38.3, 40.5, 45), alpha=0.7, color=cols[ci], label=c)
    for ax, key in zip(axes[1:], ["n_holes", "n_rim_defects", "scratch_px"]):
        jitter = np.random.default_rng(ci).uniform(-0.25, 0.25, len(sel))
        ax.scatter(ci + jitter, [f[key] for f in sel], s=10, color=cols[ci])
axes[0].axvspan(39.6, 40.4, color="#e6f4ea", zorder=0, label="tolerance 40 +/- 0.4"); axes[0].legend(fontsize=7); axes[0].set_xlabel("outer diameter (mm)")
for ax, key, thr in zip(axes[1:], ["n_holes", "n_rim_defects", "scratch_px"], [4.5, 1.5, 15]):
    ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=30, fontsize=8); ax.set_title(key)
    ax.axhline(thr, color="k", ls="--", lw=1); ax.grid(alpha=0.3)
axes[0].set_title("diameter separates undersize cleanly")
plt.suptitle("Figure 5.1 - Each feature separates one defect class from the rest (dashed: decision threshold)", fontweight="bold")
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
plot_confusion(confusion_matrix(y_te, pred_rules, len(CLASSES)), CLASSES, title=f"Rules: accuracy {acc_rules:.1%}", ax=ax)
plt.show()

## 6. The learned alternative (Week 6)

We train the **same small CNN** twice, with the same number of images:

- **end-to-end**: the raw frame, resized to 128 × 128 (the network must discover the part, its pose and the defect by itself);
- **hybrid**: the classical front end (segmentation + pose + alignment) and a **2-channel input** = aligned crop + difference from the golden template.

In [ ]:
def to_tensor_raw(X):
    g = np.stack([cv2.resize(cv2.cvtColor(x, cv2.COLOR_BGR2GRAY), (128, 128), interpolation=cv2.INTER_AREA) for x in X]).astype(np.float32)
    return torch.tensor((g / 255 - 0.45) / 0.25)[:, None]


def to_tensor_hybrid(X):
    a_ = np.stack([aligned_gray(x) for x in X])
    return torch.tensor(np.stack([(a_ / 255 - 0.45) / 0.25, (a_ - golden) / 25.0], 1))


def small_cnn(in_ch):
    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
    return nn.Sequential(block(in_ch, 16), block(16, 32), block(32, 64), block(64, 64),
                         nn.AdaptiveMaxPool2d(1), nn.Flatten(), nn.Linear(64, len(CLASSES)))


def train_eval(prep, in_ch, name):
    set_seed(0)
    t0 = time.perf_counter(); Xtr_t, Xte_t = prep(X_tr), prep(X_te); t_prep = (time.perf_counter() - t0) / (len(X_tr) + len(X_te))
    tr = DataLoader(TensorDataset(Xtr_t, torch.tensor(y_tr)), batch_size=32, shuffle=True)
    te = DataLoader(TensorDataset(Xte_t, torch.tensor(y_te)), batch_size=64)
    model = small_cnn(in_ch).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), 2e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, total_steps=EPOCHS * len(tr))
    print(f"--- {name}: {count_params(model):,} parameters")
    hist = fit(model, tr, te, opt, EPOCHS, DEVICE, scheduler=sched, amp=HW["amp"])
    logits, labels = predict(model, te, DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        model(Xte_t[:64].to(DEVICE))
    t_inf = (time.perf_counter() - t0) / min(64, len(Xte_t))
    return model, hist, logits.argmax(1), (t_prep + t_inf) * 1000


cnn_raw, hist_raw, pred_raw, ms_raw = train_eval(to_tensor_raw, 1, "end-to-end CNN on raw frames")
cnn_hyb, hist_hyb, pred_hyb, ms_hyb = train_eval(to_tensor_hybrid, 2, "hybrid: aligned crop + golden difference")
plot_history({"end-to-end (raw frame)": hist_raw, "hybrid (aligned + difference)": hist_hyb}, title="Figure 6.1 - Same CNN, same data, different input")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_confusion(confusion_matrix(y_te, pred_raw, len(CLASSES)), CLASSES, title=f"end-to-end CNN: {(pred_raw == y_te).mean():.1%}", ax=axes[0])
plot_confusion(confusion_matrix(y_te, pred_hyb, len(CLASSES)), CLASSES, title=f"hybrid CNN: {(pred_hyb == y_te).mean():.1%}", ax=axes[1])
plt.suptitle("Figure 6.2 - Confusion matrices on the test parts", fontweight="bold")
plt.tight_layout(); plt.show()

**Why the difference?** With only a few hundred images, the end-to-end network has to learn invariance to position, rotation and lighting *and*
find 1-pixel scratches. The hybrid system gives it a problem where the part is always in the same place and every deviation is already highlighted.
**Engineering knowledge reduces the amount of data a network needs.** With thousands of images and augmentation the end-to-end network would catch up,
which is exactly the trade-off studied in Week 9 (data-centric machine vision).

## 7. Engineering comparison and a robustness test

Accuracy on the development data is not the whole story. Production drifts: lamps age, exposure changes, cameras are swapped.
Below, the test parts are re-generated with **half the light and about 2.5 times the sensor noise**, without retraining or retuning anything.

In [ ]:
X_shift, y_shift, _ = partsim.make_dataset(N_TEST, seed=2, exposure=0.5, noise=8.0)
res = {}
res["rules"] = (np.array([CLASSES.index(decide_rules(features(im))) for im in X_shift]) == y_shift).mean()
for name, model, prep in [("end-to-end CNN", cnn_raw, to_tensor_raw), ("hybrid CNN", cnn_hyb, to_tensor_hybrid)]:
    lo, la = predict(model, DataLoader(TensorDataset(prep(X_shift), torch.tensor(y_shift)), batch_size=64), DEVICE)
    res[name] = (lo.argmax(1) == la).mean()
nominal = {"rules": acc_rules, "end-to-end CNN": (pred_raw == y_te).mean(), "hybrid CNN": (pred_hyb == y_te).mean()}
fig, axes = plt.subplots(1, 2, figsize=(18, 4.5))
xs = np.arange(3)
axes[0].bar(xs - 0.2, [nominal[k] for k in res], 0.4, label="nominal lighting", color="#1a73e8")
axes[0].bar(xs + 0.2, [res[k] for k in res], 0.4, label="half the light, 2.5x the noise", color="#f29900")
axes[0].set_xticks(xs); axes[0].set_xticklabels(list(res)); axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("accuracy"); axes[0].legend(); axes[0].grid(alpha=0.3, axis="y")
axes[0].set_title("robustness to a lighting change (no retuning)")
show_img = X_shift[0]
axes[1].imshow(np.hstack([X_te[0], show_img])[..., ::-1]); axes[1].axis("off"); axes[1].set_title("same part type: nominal (left) vs shifted (right)")
plt.suptitle("Figure 7.1 - Robustness test under a lighting shift", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"{'system':<18}{'nominal':>9}{'shifted':>9}{'ms/part':>9}  training images  explainable?")
for k, ms, n_img, expl in [("rules", ms_rules, 0, "yes: every decision is a measured feature"),
                           ("end-to-end CNN", ms_raw, len(y_tr), "hard: needs Grad-CAM"),
                           ("hybrid CNN", ms_hyb, len(y_tr), "partly: difference image shows where")]:
    print(f"{k:<18}{nominal[k]:>9.1%}{res[k]:>9.1%}{ms:>9.1f}  {n_img:>15}  {expl}")

**Discuss in pairs (5 min).** Look at the numbers you obtained.
1. Which system degraded most under the lighting shift, and which stage of its pipeline is responsible? (Hint: which thresholds are *fixed grey levels*?)
2. The brightness normalisation in `aligned_gray` is one line. Which Week 1–3 technique is it, and how would you add it to the rules?
3. Which system would you ship, and what would you monitor on the line?

## 8. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Outer diameter in millimetres
Return the diameter of the **minimum enclosing circle** of a contour, converted to mm with `mm_per_px`.

In [ ]:
def outer_diameter_mm(contour, mm_per_px):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    canvas = np.zeros((300, 300), np.uint8)
    cv2.circle(canvas, (150, 150), 100, 255, -1)
    cs, _ = cv2.findContours(canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    d = outer_diameter_mm(cs[0], 0.2)
    assert abs(d - 40.2) < 0.5, f"a 201 px wide disc at 0.2 mm/px should measure about 40.2 mm, got {d:.2f}"
    im, inf = partsim.make_part(3, "undersize")
    d2 = outer_diameter_mm(segment(im)[0], MM)
    assert abs(d2 - inf["outer_d_mm"]) < 0.25, f"undersize part: measured {d2:.2f} mm, true {inf['outer_d_mm']:.2f} mm"


check("E1 outer_diameter_mm", _test_e1)

### E2. Rim defects
Return how many **convexity defects** of the outer contour are deeper than `min_depth_mm`. Remember that OpenCV stores the depth as a fixed-point number (divide by 256), and that OpenCV 5 returns shape `(N, 4)` instead of `(N, 1, 4)`.

In [ ]:
def count_rim_defects(contour, mm_per_px, min_depth_mm=0.8):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    ok = [count_rim_defects(segment(partsim.make_part(s, "ok")[0])[0], MM) for s in range(5)]
    chip = [count_rim_defects(segment(partsim.make_part(s, "chip")[0])[0], MM) for s in range(5)]
    assert ok == [1] * 5, f"a good part has exactly one rim defect (the notch); got {ok}"
    assert chip == [2] * 5, f"a chipped part has two (notch + chip); got {chip}"


check("E2 count_rim_defects", _test_e2)

### E3. Your own decision rules
Write `decide(f)` for the feature dictionary returned by `features(img)` (keys `diameter_mm`, `n_holes`, `n_rim_defects`, `scratch_px`), without
copying `decide_rules`. Look at Figure 5.1 to choose the thresholds. The check requires ≥ 95% accuracy on fresh parts **and no escapes of
undersize parts** (the customer's critical defect).

In [ ]:
def decide(f):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    Xv, yv, _ = partsim.make_dataset(6 if SMOKE else 20, seed=77)
    pred = np.array([CLASSES.index(decide(features(im))) for im in Xv])
    acc = (pred == yv).mean()
    under = CLASSES.index("undersize")
    escapes = ((yv == under) & (pred == CLASSES.index("ok"))).sum()
    assert escapes == 0, f"{escapes} undersize parts were accepted as ok"
    assert acc >= 0.95, f"accuracy {acc:.1%} < 95%"


check("E3 decide", _test_e3)

### E4. A new defect appears (no self-check)
Production reports a new defect: **stains** (`partsim.make_part(seed, "stain")`, a discoloured patch). Without changing anything,
run the rules and the hybrid CNN on 50 stained parts. What do they output? Then adapt each system (a new rule vs new training data) and estimate
how long each change takes. Relate your answer to *open-set* recognition and to anomaly detection (Week 11B).

In [ ]:
# E4: your code here

### Challenge
Make the **rules** robust to the lighting shift of Section 7 without losing nominal accuracy (hint: normalise, or make the black-hat threshold
relative to the local metal brightness). Then make the **end-to-end CNN** reach ≥ 95% by changing only the *data* (more images, rotation and
brightness augmentation), not the network. Report how many images it needed.

In [ ]:
# Your challenge code here

## 9. Midterm revision (Weeks 1–6)

### 9.1 How the topics connect

In [ ]:
fig, ax = plt.subplots(figsize=(20, 7.5))
ax.set_xlim(0, 20); ax.set_ylim(0, 7.5); ax.axis("off")
nodes = {
    "W01A images:\nsampling, quantisation,\nbit depth": (2, 6.3, "#e8f0fe"), "W01B colour:\nRGB/HSV/Lab,\nhistograms": (6, 6.3, "#e8f0fe"),
    "W02A optics:\npinhole, lens,\nDOF, lighting": (10, 6.3, "#fef7e0"), "W02B calibration:\nK, distortion,\nmm per pixel": (14, 6.3, "#fef7e0"),
    "W03A point ops,\nhistogram eq.,\nhomography warp": (2, 3.8, "#e6f4ea"), "W03B convolution,\nFourier, morphology": (6, 3.8, "#e6f4ea"),
    "W04A thresholds,\nedges, Hough": (10, 3.8, "#fce8e6"), "W04B regions,\ncontours, shape": (14, 3.8, "#fce8e6"),
    "W05A corners,\nscale space,\nSIFT/ORB": (18, 3.8, "#f3e8fd"), "W05B matching,\nRANSAC, stitching": (18, 6.3, "#f3e8fd"),
    "W06A CNNs:\nconv layers as\nlearned filters": (6, 1.2, "#e0f7fa"), "W06B training,\ntransfer learning,\nGrad-CAM": (12, 1.2, "#e0f7fa"),
}
for text, (x, y, col) in nodes.items():
    ax.text(x, y, text, ha="center", va="center", fontsize=10, bbox=dict(boxstyle="round,pad=0.5", fc=col, ec="#5f6368"))
edges = [((2, 6.3), (6, 6.3), ""), ((6, 6.3), (10, 6.3), ""), ((10, 6.3), (14, 6.3), ""),
         ((2, 6.3), (2, 3.8), ""), ((2, 3.8), (6, 3.8), ""), ((6, 3.8), (10, 3.8), ""), ((10, 3.8), (14, 3.8), ""),
         ((14, 3.8), (18, 3.8), ""), ((18, 3.8), (18, 6.3), "features -> correspondences"),
         ((14, 6.3), (14, 3.8), "mm per pixel for measurement"), ((6, 3.8), (6, 1.2), "hand-made kernels -> learned kernels"),
         ((6, 1.2), (12, 1.2), "")]
for (x0, y0), (x1, y1), lab in edges:
    horizontal = abs(x1 - x0) > abs(y1 - y0)
    shrink = 62 if horizontal else 30
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color="#5f6368", lw=1.3, shrinkA=shrink, shrinkB=shrink, connectionstyle="arc3"))
    if lab:
        ax.text((x0 + x1) / 2 + (0.15 if not horizontal else 0), (y0 + y1) / 2 + (0.25 if horizontal else 0), lab,
                fontsize=8.5, color="#5f6368", ha="left" if not horizontal else "center", style="italic")
ax.text(10, 0.25, "Thread to notice: the homography of W03A (warping) returns in W05B (estimated from matches with RANSAC).",
        fontsize=9.5, color="#5f6368", style="italic", ha="center")
ax.set_title("Figure 9.1 - Concept map of Weeks 1-6", fontweight="bold", fontsize=13)
plt.show()

### 9.2 Formula sheet (know what each symbol means and when to use it)

| topic | formula |
|---|---|
| quantisation | $L = 2^b$ levels; step $\Delta = \text{range}/L$ |
| pinhole projection | $x = f\,X/Z$; with intrinsics $\tilde{\mathbf{x}} \sim K[R\,|\,\mathbf{t}]\tilde{\mathbf{X}}$, $K = \begin{bmatrix} f_x & s & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1\end{bmatrix}$ |
| radial distortion | $x_d = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$ |
| thin lens / magnification | $1/f = 1/z_o + 1/z_i$, $m = z_i/z_o$ |
| gamma | $s = c\,r^\gamma$ |
| convolution | $g(x,y) = \sum_{u,v} h(u,v)\,f(x-u, y-v)$; Fourier: $G = H \cdot F$ |
| Gaussian | $G_\sigma(x,y) = \frac{1}{2\pi\sigma^2} e^{-(x^2+y^2)/2\sigma^2}$ |
| gradient | $|\nabla I| = \sqrt{I_x^2 + I_y^2}$, $\theta = \operatorname{atan2}(I_y, I_x)$ |
| Otsu | maximise $\sigma_B^2(t) = \omega_0\omega_1(\mu_0 - \mu_1)^2$ |
| Hough (lines) | $\rho = x\cos\theta + y\sin\theta$ |
| circularity | $4\pi A / P^2$ (1 for a circle) |
| Harris | $M = \sum w \begin{bmatrix} I_x^2 & I_xI_y \\ I_xI_y & I_y^2 \end{bmatrix}$, $R = \det M - k(\operatorname{tr}M)^2$ |
| scale selection | blob of radius $r$: peak of $\sigma^2\nabla^2 G$ at $\sigma = r/\sqrt{2}$ |
| RANSAC | $N = \log(1-p)/\log(1-w^s)$ |
| conv output size | $o = \lfloor (i + 2p - k)/s \rfloor + 1$; parameters $= k^2 C_\text{in} C_\text{out} + C_\text{out}$ |
| cross-entropy | $L = -\sum_c y_c \log \hat{p}_c$, $\hat p = \operatorname{softmax}(z)$ |

### 9.3 Practice questions (discuss; no answers given here)
1. A 12-bit camera image is saved as 8-bit PNG. How many grey levels are lost, and when would this matter for inspection?
2. You must measure a 50 mm part with 0.05 mm resolution. What minimum sensor width in pixels do you need (use ≥ 2 px per smallest feature)?
3. Explain why a median filter removes salt-and-pepper noise better than a Gaussian filter of similar size.
4. Sketch the histogram of an image of dark bolts on a bright backlight. Which threshold method would you use, and when would it fail?
5. Why is the Hough transform robust to gaps in a line? What sets its angular resolution?
6. A contour has area 314 px² and perimeter 70 px. Is it closer to a circle or a square? Show your working.
7. Harris gives R < 0 at a pixel. What does that tell you about the eigenvalues of M?
8. Why does SIFT need a scale space but Harris does not? Give a case where Harris is the better choice.
9. RANSAC for a homography with 30% inliers: how many iterations for p = 0.99?
10. A 3×3 conv layer maps 64 to 128 channels. How many parameters? What is the output size of a 56×56 input with stride 2, padding 1?
11. Your CNN reaches 99% training accuracy and 70% validation accuracy. Name three things you would try, in order.
12. In the inspection lab above, which stage would you change first if the conveyor colour changed from black to white? Why?

### 9.4 Revision quiz (auto-marked)

In [ ]:
quiz([
    {
        "q": 'An 8-bit greyscale image has how many possible intensity levels?',
        "options": [
            '8',
            '128',
            '256',
            '65536',
        ],
        "answer": '5aae995772',
        "explain": '2^8 = 256.',
    },
    {
        "q": 'Which colour space separates hue from brightness, making colour thresholds more robust to lighting?',
        "options": [
            'RGB',
            'HSV',
            'Bayer',
            'YUV 4:2:0',
        ],
        "answer": '5839999511',
        "explain": 'Hue is (mostly) independent of intensity; Lab is another good choice.',
    },
    {
        "q": 'Which lighting technique best reveals the silhouette of a part for measurement?',
        "options": [
            'Diffuse dome light',
            'Backlight',
            'Dark-field ring light',
            'Coaxial light',
        ],
        "answer": '2a4369e83e',
        "explain": 'A backlight gives a high-contrast silhouette.',
    },
    {
        "q": 'What does the intrinsic matrix K contain?',
        "options": [
            'Rotation and translation',
            'Focal lengths and principal point',
            'Distortion coefficients only',
            'The homography',
        ],
        "answer": '9b0c6f0691',
        "explain": 'Extrinsics R, t are separate; distortion coefficients are separate too.',
    },
    {
        "q": 'Which filter preserves edges while removing noise?',
        "options": [
            'Box filter',
            'Gaussian filter',
            'Bilateral filter',
            'Laplacian',
        ],
        "answer": 'fd62242258',
        "explain": 'It weights neighbours by both distance and intensity similarity (the median also preserves edges).',
    },
    {
        "q": 'Morphological opening (erosion then dilation) is mainly used to...',
        "options": [
            'fill small holes',
            'remove small bright specks',
            'detect edges',
            'equalise the histogram',
        ],
        "answer": '1581388e17',
        "explain": 'Closing (dilation then erosion) fills small holes.',
    },
    {
        "q": "Otsu's method chooses the threshold that...",
        "options": [
            'maximises the between-class variance',
            'minimises the image entropy',
            'equals the mean',
            'maximises the number of edges',
        ],
        "answer": '6303498913',
        "explain": 'Equivalently, it minimises the within-class variance.',
    },
    {
        "q": 'A perfect circle has circularity 4*pi*A/P^2 equal to...',
        "options": [
            '0',
            '0.785',
            '1',
            'pi',
        ],
        "answer": '3165be1bd7',
        "explain": 'Any other shape gives a smaller value.',
    },
    {
        "q": 'Which detector gives each keypoint its own characteristic scale?',
        "options": [
            'Harris',
            'FAST',
            'Shi-Tomasi',
            'SIFT',
        ],
        "answer": 'b32d2f87c0',
        "explain": 'Extrema of the DoG over position and scale.',
    },
    {
        "q": 'Why is the ratio test used when matching descriptors?',
        "options": [
            'To speed up matching',
            'To reject ambiguous matches',
            'To make descriptors binary',
            'To compute the homography',
        ],
        "answer": 'cfe47726e5',
        "explain": 'A correct match is much closer than the second-best candidate.',
    },
    {
        "q": 'How many parameters does a Conv2d(3, 16, kernel_size=3) layer have (with bias)?',
        "options": [
            '48',
            '144',
            '448',
            '432',
        ],
        "answer": '4dbca0aaf1',
        "explain": '3*3*3*16 + 16 = 448.',
    },
    {
        "q": 'Transfer learning is most useful when...',
        "options": [
            'you have millions of labelled images',
            'you have a small dataset for a new task',
            'the images are not natural images',
            'you want to avoid GPUs',
        ],
        "answer": 'd79ccbd72a',
        "explain": 'Pretrained features transfer; only the head (or a little more) is trained.',
    },
    {
        "q": 'In the lab, why did the hybrid CNN need far less data than the end-to-end CNN?',
        "options": [
            'It had more parameters',
            'The classical front end removed pose and lighting variation and highlighted defects',
            'It trained for longer',
            'It used colour',
        ],
        "answer": 'aae3994934',
        "explain": 'Domain knowledge reduces the variation the network must learn.',
    },
    {
        "q": 'Which error is usually most costly in quality inspection?',
        "options": [
            'False reject of a good part',
            'Escape of a defective part',
            'Both equal',
            'Neither',
        ],
        "answer": '2384f4e9e1',
        "explain": 'Escapes reach the customer; thresholds are usually set to minimise them.',
    },
], title='Midterm revision quiz (Weeks 1-6)')

## 10. Exit ticket (reflection)
1. Which single week's material mattered most for today's system, and why?
2. Name one assumption of the rule-based system that would break on a different production line.
3. What is one topic from Weeks 1–6 you want to revise before the midterm?

## 11. What's new (2025–26) in industrial inspection

> - **Hybrid pipelines are the norm.** Commercial tools (e.g. MVTec HALCON, Cognex VisionPro Deep Learning, Keyence) combine calibrated
>   measurement and matching with deep-learning classification, segmentation and anomaly detection in one flow, as you did today.
> - **Anomaly detection trained on good parts only** (PatchCore, EfficientAD; open-source in *Anomalib*) handles defects that were never seen
>   in training, such as the stain in E4. Week 11B covers it.
> - **Synthetic data**: procedurally generated or rendered parts (as `partsim` does) and diffusion-generated defects are used to bootstrap models before
>   real defects are available (Week 13B).
> - **Edge deployment**: models are exported to ONNX / TensorRT / OpenVINO and run on the line PC or smart camera (Week 9B).

## 12. References and further exploration
- Steger, C., Ulrich, M. & Wiedemann, C. (2018). *Machine Vision Algorithms and Applications*, 2nd ed. Wiley. (Ch. 3: measurement, template matching, industrial practice.)
- Batchelor, B. G. (Ed.) (2012). *Machine Vision Handbook.* Springer. [doi:10.1007/978-1-84996-169-1](https://doi.org/10.1007/978-1-84996-169-1)
- Bergmann, P. et al. (2019). *MVTec AD — A comprehensive real-world dataset for unsupervised anomaly detection.* CVPR. [doi:10.1109/CVPR.2019.00982](https://doi.org/10.1109/CVPR.2019.00982)
- Roth, K. et al. (2022). *Towards total recall in industrial anomaly detection* (PatchCore). CVPR. [arXiv:2106.08265](https://arxiv.org/abs/2106.08265)
- Akcay, S. et al. (2022). *Anomalib: A deep learning library for anomaly detection.* [arXiv:2202.08341](https://arxiv.org/abs/2202.08341)
- Ng, A. (2021). *A chat with Andrew on MLOps: from model-centric to data-centric AI.* DeepLearning.AI talk (search the title on YouTube); the idea is developed in Week 9A.
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., Ch. 3–7 (revision). [szeliski.org/Book](https://szeliski.org/Book/)
- OpenCV: *Contours hierarchy*, *Convexity defects*, *Morphological transformations*. [docs.opencv.org](https://docs.opencv.org/4.x/d3/d05/tutorial_py_table_of_contents_contours.html)

**Data:** every image in this notebook is generated by `resources/scripts/partsim.py` (no external data).

## 13. Key takeaways
- Start from the **specification**: tolerances, defect list, throughput and the cost of each error type.
- A working system chains **all** the basics: acquisition, pre-processing, segmentation, pose from features, alignment, calibrated measurement.
- **Rules** are fast, need no training data and are explainable, but fixed thresholds can be brittle.
- A CNN on raw frames needs much more data; a **hybrid** (classical front end + learned decision) is accurate with little data.
- Always test **robustness** under the changes you expect on the line, and report escapes and false rejects separately.

**Next session (07B): midterm.** After the break: object detection (Week 8).